# Organizing text using the latent space

## Outline

1.   Install libraries and download data
2.   Extract text features/embeddings
3.   Reduce dimensionality using UMAP
4.   Create an interactive plot of the texts


#Setup


In [ ]:
#@title ▶ Install the required tools

!pip install -q sentence_transformers
!pip install -q umap-learn
!pip install -q lap scipy
!pip install -q datasets

In [ ]:
#@title ▶ Download the embedding model

from sentence_transformers import SentenceTransformer, util

embedding_model = SentenceTransformer("clip-ViT-B-32")

In [ ]:
#@title ▶ Load a dataset (from [Huggingface](https://huggingface.co/datasets))

from datasets import load_dataset

dataset = load_dataset("valhalla/emoji-dataset")

In [ ]:
#@title ▶ Extract the images

images = dataset['train']['image']
texts = dataset['train']['text']

In [ ]:
#@title Uncomment to connect drive

# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
#@title ▶ [Alternative if you have your own images] Load the images from a folder

#@markdown ⚠ This may not work if you have a big amount of big images.

#@markdown Check use_own to use your files instead of huggingface dataset.
use_own = False #@param {type:"boolean"}

#@markdown Select the path to your files.

#@markdown The best way is to get the path is find it in the file explorer on the left, click on the 3 points, use the option "copy path" and paste the value here.

#@markdown ⚠ All the images must be in jpg, jpeg or png format.
input_path = "/content/drive/MyDrive/" #@param {type:"string"}

import os
import glob
from PIL import Image
from tqdm import tqdm

if use_own:
  images_list = sorted(glob.glob(os.path.join(input_path, "*.jpg")) + glob.glob(os.path.join(input_path, "*.png")))
  images = []
  texts = None
  for image_path in tqdm(images_list):
    images.append(Image.open(image_path))


#Processing

In [ ]:
#@title ▶ Calulate the position of the images in the latent space (calculate the embedding) (*)

embeddings = embedding_model.encode(images, convert_to_tensor=True, show_progress_bar=True)

print(embeddings.shape)

In [ ]:
#@title ▶ Convert from 384 dimensions to 2 dimensions

from umap import UMAP

model = UMAP(
    n_components=2,
    metric='cosine')
embeddings_2d = model.fit_transform(embeddings.cpu())

print(embeddings_2d.shape)

#Plotting

In [ ]:
#@title ▶ Create an interactive chart

import textwrap
import plotly.express as px

if 'texts' in vars() and texts is not None:
  width = 48
  wrapped_texts = ["<br>".join(textwrap.wrap(text, width, break_long_words=False)) for text in texts]

  fig = px.scatter(hover_name=wrapped_texts, x=embeddings_2d[:,0], y=embeddings_2d[:,1])
else:
  fig = px.scatter(x=embeddings_2d[:,0], y=embeddings_2d[:,1])

fig.show()

# Creating an image matrix

In [ ]:
#@title ▶ Convert from 2D positions to 2D grid (*)

#@markdown It also displays the calculated positions in a 2D grid

import math
from scipy.spatial.distance import cdist
import numpy as np
import lap
from matplotlib import pyplot as plt

def get_grid_layout(data_2d):
  side = math.ceil(math.sqrt(data_2d.shape[0]))
  width = side
  height = side if side * (side - 1) < data_2d.shape[0] else side - 1

  xv, yv = np.meshgrid(np.linspace(0, 1, width), np.linspace(0, 1, height))
  grid = np.dstack((xv, yv)).reshape(-1, 2)

  cost = cdist(grid, data_2d, 'sqeuclidean')
  cost = cost * (10000000. / cost.max())

  # # # usant cost.astype(int) en principi és més ràpid, però no ens importa
  min_cost, row_assigns, col_assigns = lap.lapjv(cost, extend_cost=True)
  return width, height, grid[col_assigns]

embeddings_2d -= embeddings_2d.min(axis=0)
embeddings_2d /= embeddings_2d.max(axis=0)

width, height, grid = get_grid_layout(embeddings_2d)

plt.figure(figsize=(8,8))
plt.scatter(grid[:,0], grid[:,1], marker='o', s=12)
plt.show()



In [ ]:
#@title 🖼 Generate image matrix

import os
from PIL import Image, ImageDraw
from google.colab import files

def generate_matrix(grid_xy, images, width, height, image_size, proportion_w, proportion_h, margin, background_color, swap=False):
  pos_x = 0
  pos_y = 1
  if swap:
    pos_x = 1
    pos_y = 0

  tile_size_dst_w = image_size
  tile_size_dst_h = math.floor(tile_size_dst_w * proportion_h / proportion_w)

  image_width = width * (tile_size_dst_w + 2 * margin)
  image_height = height * (tile_size_dst_h + 2 * margin)

  im = Image.new("RGB", (image_width, image_height))

  draw = ImageDraw.Draw(im)
  draw.rectangle(((0, 0), (image_width, image_height)), fill=f'#{background_color}')

  for idx, image in enumerate(images):
      pos = grid_xy[idx]

      corr_pos = [pos[0]*(width-1)/width, pos[1]*(height-1)/height]

      displ_x = tile_size_dst_w + 2 * margin;
      displ_y = tile_size_dst_h + 2 * margin

      left = int(margin + corr_pos[0]*image_width)
      top = int(margin + corr_pos[1]*image_height)

      tile_im = image.copy()
      tile_im.thumbnail((tile_size_dst_w, tile_size_dst_h), Image.Resampling.LANCZOS)
      tile_w, tile_h = tile_im.size
      tile_disp_w = math.floor((tile_size_dst_w - tile_w)/2)
      tile_disp_h = math.floor((tile_size_dst_h - tile_h)/2)
      im.paste(tile_im, (left + tile_disp_w, top + tile_disp_h))

  output_image = os.path.join("/content", f'grid.jpg')
  im.save(output_image, quality=90)
  files.download(output_image)

  im.resize((min(1920, image_width), min(1920, image_height)))
  display(im)

#@markdown Individual image size. Big values make the grid image bigger and may fail.
image_size = 64 #@param {type:"number"}

#@markdown Individual image proportion. Leave at 1/1 if images have diferent orientations or 16/9 if they are frames from a video.
proportion_w = 1 #@param {type:"number"}
proportion_h = 1 #@param {type:"number"}

#@markdown Margin between images (in pixels)
margin = 0 #@param {type:"number"}

#@markdown Background color in hexadecimal format (you can copy/paste from photoshop). 000000 is black, ffffff is white
background_color = "ffffff" #@param {type:"string"}

generate_matrix(grid, images, width, height, image_size, proportion_w, proportion_h, margin, background_color)


# Credits

Taller Estampa https://tallerestampa.com / https://github.com/estampa
